# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described by a Croissant schema located at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

You will learn to:
- Load Croissant dataset objects
- Inspect entities (record sets, fields, columns) by their `@id`
- Extract and process tabular data
- Perform basic exploratory data analysis (EDA) and visualization

In [ ]:
# Install mlcroissant if not already present
!pip install -q mlcroissant

## 1. Data Loading
Load dataset and metadata using the Croissant schema URL. `mlcroissant` retrieves schema information and exposes the dataset's hierarchical structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Get top-level dataset metadata (single object, not dict)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Let's explore the `recordSet` structure and list all record sets, their `@id`s, and the fields/columns they contain. All entities below are referenced strictly by their `@id` as per best practice.

We will enumerate top-level record sets, then for each, enumerate their `field`s and/or `column`s by `@id`.

In [ ]:
# List all record sets (@id) with their field and column @ids
if not dataset.record_sets:
    print("No record sets explicitly defined in metadata. Trying to infer from distribution...")
    # Attempt to infer record sets from data distributions, for demonstration
    for i, d in enumerate(getattr(meta, 'distribution', [])):
        print(f"Distribution {i}: @id = {getattr(d, '@id', str(d))}")
else:
    for rs in dataset.record_sets:
        print(f"RecordSet: @id = {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict): fields = [fields]
        for field in fields:
            print(f"  Field: @id = {field['@id']}")
        columns = rs.get('column', [])
        if isinstance(columns, dict): columns = [columns]
        for col in columns:
            print(f"  Column: @id = {col['@id']}")

# Demonstrate how to iterate records for a record set by @id (replace as_necessary_for_your_data)
example_record_set_id = None
if dataset.record_sets:
    example_record_set_id = dataset.record_sets[0]['@id']
    print(f"\nSample records (first 2) from record set @id = {example_record_set_id}:")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i == 1: break
else:
    print("No record sets found by @id.")

## 3. Data Extraction

Let's load records from all available record sets by their `@id` into `pandas.DataFrame` objects. Columns and fields will be shown using their `@id` values as well.

*If the dataset uses only one main record set, we'll extract that one for demonstration.*

In [ ]:
# We'll collect all record set @id's found
record_set_ids = []
if dataset.record_sets:
    for rs in dataset.record_sets:
        record_set_ids.append(rs['@id'])
else:
    # For demonstration if record_sets are not defined, attempt inference from distribution
    if hasattr(meta, 'distribution'):
        for d in getattr(meta, 'distribution', []):
            rs_id = getattr(d, '@id', None)
            if rs_id: record_set_ids.append(rs_id)
if not record_set_ids:
    print("No record set @id's found to extract data.")

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '@id': {rs_id}")
        print(f"Fields/Columns by @id: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for record set @id={rs_id}: {e}")

# Choose the first available DataFrame (for demo)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nWorking with table from record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We select a numerical field (by its `@id`) for demonstration, filter and normalize it, and group by another attribute (`@id`).

Replace `<numeric_field_id>` and `<group_field_id>` below with actual field/column `@id`s for the dataset you loaded. For this notebook, we'll attempt to infer appropriate fields.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to find a numeric column by @id automatically
    numeric_field_id = None
    for col in df.columns:
        # Try first float/int column
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.90) if df[numeric_field_id].nunique() > 10 else df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize this numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to pick a group-by field with a small number of categories
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < 10 and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print("Grouped summary:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("Cannot perform EDA: No record set DataFrame available.")

## 5. Visualization

Visualizing the distribution of a numeric field or the relationship between two fields. This cell will plot the selected numeric field and, if possible, a group breakdown.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes and (numeric_field_id is not None):
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].plot(kind='hist', bins=20, alpha=0.7)
    plt.title(f"Distribution of numeric field (@id: {numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
    # If a group_field_id exists, boxplot by group
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load Croissant metadata and records from the dataset URL
- Discover and reference all record sets and fields/columns by their stable `@id`
- Extract records into dataframes for tabular data analysis
- Perform simple EDA and visualize data distributions

You can now extend this workflow for deeper analysis, customized filtering, advanced visualization, or downstream ML tasks, always referencing data entities by their `@id` as required for robust, reproducible research.